# Module 05 — Multiplicative Inverses Modulo *n*
### Mathematics of Cryptography · Python Companion Notebook

This notebook lets you **verify, find, and use** multiplicative inverses modulo *n* with working Python code.  
Every tool mirrors the examples in Tutorial 05. Run each cell top-to-bottom.

| Section | Topic |
|---|---|
| 1 | Setup and display helpers |
| 2 | Verifying an inverse claim |
| 3 | Finding inverses by search |
| 4 | The GCD existence rule |
| 5 | Visualising the units of ℤₙ |
| 6 | Affine cipher — encrypt and decrypt |
| 7 | Exercises |
| 8 | Summary and what comes next |

---
## Section 1 — Setup and Display Helpers

In [ ]:
import math

def separator(title=''):
    width = 60
    if title:
        pad = (width - len(title) - 2) // 2
        print('─' * pad + f' {title} ' + '─' * (width - pad - len(title) - 2))
    else:
        print('─' * width)

def show_verify(a, x, n):
    """Show whether x is the multiplicative inverse of a modulo n."""
    product = a * x
    residue = product % n
    verdict = '✓ YES — valid inverse' if residue == 1 else '✗ NO — not an inverse'
    print(f'  a = {a},  x = {x},  n = {n}')
    print(f'  {a} × {x} = {product}')
    print(f'  {product} mod {n} = {residue}')
    print(f'  → {verdict}')

print('Helpers loaded.')

---
## Section 2 — Verifying an Inverse Claim

The definition is simple: *x* is the multiplicative inverse of *a* modulo *n* when

$$ax \equiv 1 \pmod{n}$$

To verify a claim, multiply and reduce.

In [ ]:
separator('Tutorial example: 5⁻¹ ≡ 21 (mod 26)')
show_verify(5, 21, 26)

separator('True or false: 3⁻¹ ≡ 7 (mod 10)?')
show_verify(3, 7, 10)

separator('True or false: 4⁻¹ ≡ 4 (mod 10)?')
show_verify(4, 4, 10)

### Exercise 2.1

Verify each claim before running:
1. Is 7 the inverse of 7 modulo 12?
2. Is 3 the inverse of 9 modulo 26?
3. Is 2 the inverse of 14 modulo 26?

In [ ]:
# Your work here — call show_verify(a, x, n) for each


---
## Section 3 — Finding Inverses by Search

For small *n*, brute force works: test every residue *x* in {0, 1, …, n−1} until *ax* mod *n* = 1.

In [ ]:
def find_inverse_search(a, n, verbose=True):
    """Find the multiplicative inverse of a modulo n by exhaustive search.
    Returns the inverse, or None if no inverse exists."""
    if verbose:
        separator(f'Search: {a}⁻¹ mod {n}')
        print(f'  {'x':>4}  {'a·x':>8}  {'a·x mod n':>10}  {'Hit?':>6}')
        print(f'  {"─"*4}  {"─"*8}  {"─"*10}  {"─"*6}')
    for x in range(1, n):
        product = a * x
        residue = product % n
        hit = residue == 1
        if verbose:
            marker = ' ← FOUND' if hit else ''
            print(f'  {x:>4}  {product:>8}  {residue:>10}  {"Yes" if hit else "":>6}{marker}')
        if hit:
            if verbose:
                print(f'\n  Result: {a}⁻¹ ≡ {x} (mod {n})')
            return x
    if verbose:
        print(f'\n  No inverse exists for {a} mod {n}.')
    return None

find_inverse_search(5, 26)
print()
find_inverse_search(3, 10)

In [ ]:
# Try a number with no inverse
find_inverse_search(2, 6)

### Exercise 3.1

Use `find_inverse_search` to find:
1. `11⁻¹ mod 26`
2. `7⁻¹ mod 12`
3. `6⁻¹ mod 9` — predict the result before running

In [ ]:
# Your work here


---
## Section 4 — The GCD Existence Rule

Testing every residue works for small *n*, but cryptography uses moduli like 2¹⁰²⁴. We need a faster test.

> **Rule:** The inverse of *a* modulo *n* exists **if and only if** gcd(*a*, *n*) = 1.

Numbers with gcd = 1 are called **coprime**. Numbers with a modular inverse are called **units** modulo *n*.

In [ ]:
def inverse_exists(a, n):
    """Return True if a has a multiplicative inverse modulo n."""
    return math.gcd(a, n) == 1

separator('GCD rule — existence check')
pairs = [(5, 26), (2, 6), (4, 10), (7, 12), (13, 26), (15, 26), (9, 26)]
print(f'  {'a':>4}  {'n':>4}  {'gcd':>5}  {'Inverse exists?':>16}')
print(f'  {"─"*4}  {"─"*4}  {"─"*5}  {"─"*16}')
for a, n in pairs:
    g = math.gcd(a, n)
    exists = 'Yes' if g == 1 else 'No'
    print(f'  {a:>4}  {n:>4}  {g:>5}  {exists:>16}')

In [ ]:
# Python's built-in pow(a, -1, n) computes the inverse directly (Python 3.8+)
separator('Built-in: pow(a, -1, n)')
for a, n in [(5, 26), (3, 10), (7, 12), (11, 26)]:
    try:
        inv = pow(a, -1, n)
        print(f'  {a}⁻¹ mod {n} = {inv}   (verify: {a}×{inv} mod {n} = {(a*inv)%n})')
    except ValueError:
        print(f'  {a}⁻¹ mod {n} does not exist   (gcd = {math.gcd(a,n)})')

### Exercise 4.1

Without running code first, predict whether each inverse exists. Then verify:
1. `17 mod 26`
2. `12 mod 26`
3. `14 mod 15`
4. `9 mod 15`

In [ ]:
# Your predictions as comments, then verify with inverse_exists() or pow()


---
## Section 5 — Visualising the Units of ℤₙ

The **units** of ℤₙ are all residues in {1, …, n−1} that have a multiplicative inverse.  
The count of units is given by Euler's totient function φ(n).

In [ ]:
def units_of(n):
    """Return the list of units (invertible elements) modulo n."""
    return [a for a in range(1, n) if math.gcd(a, n) == 1]

def show_units(n):
    units = units_of(n)
    non_units = [a for a in range(1, n) if math.gcd(a, n) != 1]
    separator(f'Units of ℤ_{n}')
    print(f'  Units ({len(units)}):     {units}')
    print(f'  Non-units ({len(non_units)}):  {non_units}')
    print(f'  φ({n}) = {len(units)}')

for n in [6, 10, 12, 26]:
    show_units(n)
    print()

In [ ]:
# When n is prime, every non-zero element is a unit
separator('When n is prime: all non-zero elements are units')
for n in [7, 11, 13, 17]:
    units = units_of(n)
    print(f'  n={n}: {len(units)} units out of {n-1} non-zero elements — φ({n}) = {n-1}')

### Exercise 5.1

1. How many units does ℤ₂₆ have? List them.
2. How many units does ℤ₂₅₆ have? (Think about what 256 = 2⁸ means for coprimality.)
3. How many units does ℤ₂₅₇ have? (257 is prime.)

In [ ]:
# Your work here


---
## Section 6 — Affine Cipher: Encrypt and Decrypt

The affine cipher maps each letter number *P* ∈ {0, …, 25} to a ciphertext letter number *C*:

$$C \equiv aP + b \pmod{26}$$

Decryption requires the inverse of *a* modulo 26:

$$P \equiv a^{-1}(C - b) \pmod{26}$$

For decryption to work, *a* **must** be a unit modulo 26 — i.e., gcd(*a*, 26) = 1.

In [ ]:
LETTERS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

def letter_to_num(ch):
    return LETTERS.index(ch.upper())

def num_to_letter(n):
    return LETTERS[n % 26]

def affine_encrypt(plaintext, a, b):
    """Encrypt plaintext with affine cipher C = (a*P + b) mod 26."""
    if math.gcd(a, 26) != 1:
        raise ValueError(f'a={a} has no inverse mod 26 — cannot decrypt')
    result = []
    for ch in plaintext.upper():
        if ch in LETTERS:
            P = letter_to_num(ch)
            C = (a * P + b) % 26
            result.append(num_to_letter(C))
        else:
            result.append(ch)
    return ''.join(result)

def affine_decrypt(ciphertext, a, b):
    """Decrypt ciphertext with affine cipher P = a_inv * (C - b) mod 26."""
    a_inv = pow(a, -1, 26)
    result = []
    for ch in ciphertext.upper():
        if ch in LETTERS:
            C = letter_to_num(ch)
            P = (a_inv * (C - b)) % 26
            result.append(num_to_letter(P))
        else:
            result.append(ch)
    return ''.join(result)

separator('Affine cipher example: a=5, b=8')
a, b = 5, 8
plaintext = 'MULTIPLICATIVE INVERSES'
ct = affine_encrypt(plaintext, a, b)
pt = affine_decrypt(ct, a, b)
a_inv = pow(a, -1, 26)
print(f'  Key:        a={a}, b={b}')
print(f'  a⁻¹ mod 26: {a_inv}   (verify: {a}×{a_inv} mod 26 = {(a*a_inv)%26})')
print(f'  Plaintext:  {plaintext}')
print(f'  Ciphertext: {ct}')
print(f'  Decrypted:  {pt}')

In [ ]:
# Show why a must be a unit — what breaks when gcd(a,26) != 1
separator('Bad key: a=13 (gcd(13,26)=13 — not a unit)')
a_bad = 13
print(f'  gcd(13,26) = {math.gcd(13,26)}')
print(f'  Trying to encrypt A, B, C, D, ...')
outputs = {}
for ch in LETTERS:
    P = letter_to_num(ch)
    C = (a_bad * P + 0) % 26
    outputs.setdefault(C, []).append(ch)
collisions = {k: v for k, v in outputs.items() if len(v) > 1}
print(f'  Collisions found: {len(collisions)}')
for c, letters in list(collisions.items())[:3]:
    print(f'    Letters {letters} all map to ciphertext {LETTERS[c]}')
print('  → Decryption is impossible: multiple plaintexts share the same ciphertext.')

### Exercise 6.1

1. Encrypt `'INVERSE'` with key *a*=7, *b*=3. Then decrypt the result to verify.
2. What is the inverse of 7 modulo 26? Confirm with `pow(7, -1, 26)`.
3. Try to encrypt with *a*=2. What error do you get and why?

In [ ]:
# Your work here


### Exercise 6.2 — Crack the affine cipher

The message below was encrypted with an affine cipher.  
You know that *b* = 4 and that the first plaintext letter is **I** (number 8).

In [ ]:
mystery = 'YCNJLYXCIJ YNJLYXIJ'
b_known = 4
first_plain = 8   # letter I

# Hint: the first ciphertext letter tells you what a * first_plain + b ≡ (mod 26).
# From that, you can determine a, then test with the units of Z_26.

# Your work here — find a, then call affine_decrypt(mystery, a, b_known)


---
## Section 7 — Exercises

### Exercise 7.1 — Full inverse table for ℤ₂₆

Build a complete table showing every unit of ℤ₂₆ and its inverse.

In [ ]:
separator('Inverse table for ℤ₂₆')
print(f'  {'a':>4}  {'a⁻¹ mod 26':>12}  {'verify (a × a⁻¹) mod 26':>24}')
print(f'  {"─"*4}  {"─"*12}  {"─"*24}')
for a in range(1, 26):
    if math.gcd(a, 26) == 1:
        inv = pow(a, -1, 26)
        check = (a * inv) % 26
        print(f'  {a:>4}  {inv:>12}  {check:>24}')

### Exercise 7.2 — Self-inverse numbers

A number *a* is **self-inverse** modulo *n* if *a*² ≡ 1 (mod *n*), i.e., *a*⁻¹ = *a*. Find all self-inverse elements of ℤ₂₆.

In [ ]:
# Your work here — find all a in 1..25 where (a*a) % 26 == 1


### Exercise 7.3 — Modular inverse for large numbers

`pow(a, -1, n)` works instantly even for very large *n*. Try it with RSA-scale numbers.

In [ ]:
# A small RSA-like example
# p and q are prime, n = p*q, e must be coprime to (p-1)*(q-1)
p, q = 61, 53
n = p * q
phi_n = (p - 1) * (q - 1)   # Euler's totient
e = 17                        # public exponent

print(f'n = {p} × {q} = {n}')
print(f'φ(n) = {p-1} × {q-1} = {phi_n}')
print(f'e = {e},  gcd(e, φ(n)) = {math.gcd(e, phi_n)}')

d = pow(e, -1, phi_n)   # private key exponent — the modular inverse!
print(f'd = e⁻¹ mod φ(n) = {d}')
print(f'Verify: e × d mod φ(n) = {(e * d) % phi_n}')

---
## Section 8 — Summary and What Comes Next

### What you have built

| Tool | What it does |
|---|---|
| `show_verify(a, x, n)` | Checks whether *x* is the inverse of *a* mod *n* |
| `find_inverse_search(a, n)` | Finds *a*⁻¹ mod *n* by exhaustive search with a printed table |
| `inverse_exists(a, n)` | Returns True iff gcd(*a*, *n*) = 1 |
| `pow(a, -1, n)` | Python built-in — finds inverse instantly for any size *n* |
| `units_of(n)` | Lists all units (invertible elements) of ℤₙ |
| `affine_encrypt / decrypt` | Full affine cipher over the 26-letter alphabet |

### Key facts to remember

- *x* is the multiplicative inverse of *a* mod *n* when *ax* ≡ 1 (mod *n*)  
- An inverse exists **if and only if** gcd(*a*, *n*) = 1  
- When *n* is prime, **every** nonzero element has an inverse  
- The inverse depends on the modulus — it changes with *n*  
- Python's `pow(a, -1, n)` uses the **extended Euclidean algorithm** internally

---
## What Comes Next

**Module 06 — The Euclidean Algorithm** shows *why* `pow(a, -1, n)` works.  
The extended Euclidean algorithm computes gcd(*a*, *n*) and, along the way, finds integers *s*, *t* such that *as* + *nt* = 1.  
That *s* is exactly *a*⁻¹ mod *n*. You will implement it from scratch.